# ⚡ Spark Streaming — Ingest Kafka to HDFS Bronze Layer

This notebook runs a **Spark Structured Streaming** job that:
1. Connects to the **Spark Master** cluster (`spark-master:7077`).
2. Consumes messages from Kafka topic `trips-raw`.
3. Parses the NYC Taxi trip records.
4. Writes raw data as **Parquet** files into **HDFS** (`/hdfs/bronze/yellow_trips`).

### Architecture
```
Kafka: trips-raw (3 Partitions)
   ├── Partition 0 ──► Spark Worker 1 (ahmed)
   ├── Partition 1 ──► Spark Worker 2 (momen ayman)
   └── Partition 2 ──► Spark Worker 3 (mohamostafa)
                           ▼
           HDFS DataNodes → /hdfs/bronze/yellow_trips/
```

---
## 1. Initialize Spark Session

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)

# ── Configuration ────────────────────────────────────────────────────────
KAFKA_BOOTSTRAP = "kafka:9092"
KAFKA_TOPIC     = "trips-raw"
MASTER_HOST     = os.getenv("MASTER_HOST", "100.121.51.9")
HDFS_URI        = f"hdfs://{MASTER_HOST}:9000"
BRONZE_PATH     = f"{HDFS_URI}/hdfs/bronze/yellow_trips"
CHECKPOINT_PATH = f"{HDFS_URI}/hdfs/checkpoints/bronze_yellow_trips"

# JARs are pre-placed on the shared spark-apps volume → Workers load them locally
# No JAR transfer over Tailscale network!
WORKER_JARS_PATH = "/opt/spark-apps"
JAR_NAMES = [
    "org.apache.spark_spark-sql-kafka-0-10_2.12-3.3.0.jar",
    "org.apache.kafka_kafka-clients-2.8.1.jar",
    "org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.3.0.jar",
    "org.apache.commons_commons-pool2-2.11.1.jar",
    "org.apache.hadoop_hadoop-client-api-3.3.2.jar",
    "org.apache.hadoop_hadoop-client-runtime-3.3.2.jar",
    "org.lz4_lz4-java-1.8.0.jar",
    "org.xerial.snappy_snappy-java-1.1.8.4.jar",
    "org.slf4j_slf4j-api-1.7.32.jar",
    "commons-logging_commons-logging-1.1.3.jar",
    "com.google.code.findbugs_jsr305-3.0.0.jar",
]

# Build executor classpath (worker-local paths)
EXECUTOR_CLASSPATH = ":".join([f"{WORKER_JARS_PATH}/{j}" for j in JAR_NAMES])
# Build driver jars (local paths inside jupyter container)
DRIVER_JARS = ",".join([f"/home/jovyan/work/spark-apps/{j}" for j in JAR_NAMES])

print(f"📡 Kafka Bootstrap : {KAFKA_BOOTSTRAP}")
print(f"📦 Target HDFS Path: {BRONZE_PATH}")
print(f"🔧 JARs: loaded from shared volume (no network transfer!)")

# Stop any old active SparkSession first
try:
    spark.stop()
    print("🔄 Stopped old SparkSession.")
except NameError:
    pass

# ── SparkSession Builder ──────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("NYC_Taxi_Kafka_To_HDFS_Bronze")
    .master("spark://spark-master:7077")
    # Driver loads JARs from its own container path
    .config("spark.jars", DRIVER_JARS)
    # Executors load JARs from their local /opt/spark-apps path (no network!)
    .config("spark.executor.extraClassPath", EXECUTOR_CLASSPATH)
    # Don't upload JARs to workers — they already have them locally
    .config("spark.executor.userClassPathFirst", "true")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.driver.host", MASTER_HOST)
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7079")
    .config("spark.blockManager.port", "7076")
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .config("spark.hadoop.dfs.datanode.use.datanode.hostname", "true")
    .config("spark.hadoop.dfs.replication", "1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark Session initialized successfully!")
print(f"🔗 Spark UI: http://{MASTER_HOST}:8080")

---
## 2. Define NYC Taxi Data Schema

In [ ]:
taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
])

print("📋 Schema defined with 19 columns.")

---
## 3. Read Stream from Kafka

In [ ]:
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

parsed_stream = (
    kafka_stream
    .selectExpr("CAST(value AS STRING) as json_payload", "timestamp as kafka_ingest_time")
    .select(
        from_json(col("json_payload"), taxi_schema).alias("data"),
        col("kafka_ingest_time")
    )
    .select("data.*", "kafka_ingest_time")
    .withColumn("hdfs_ingest_time", current_timestamp())
)

print("🌊 Streaming dataframe created. Schema:")
parsed_stream.printSchema()

---
## 4. Write Stream to HDFS Bronze (Parquet)

In [ ]:
query = (
    parsed_stream.writeStream
    .format("parquet")
    .option("path", BRONZE_PATH)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

print("🚀 Micro-batch Streaming job started across Spark Workers!")
print("⏳ Waiting for batch completion...")

query.awaitTermination()

print("✅ Processing complete! Data successfully written to HDFS Bronze Layer.")

---
## 5. Verify HDFS Data Distribution

In [ ]:
bronze_df = spark.read.parquet(BRONZE_PATH)

total_records = bronze_df.count()
print(f"📊 Total Records in HDFS Bronze: {total_records:,}")
print(f"📁 Partition File Count: {bronze_df.rdd.getNumPartitions()}")

print("\n── Sample Records from HDFS ──")
bronze_df.select("VendorID", "tpep_pickup_datetime", "trip_distance", "total_amount", "hdfs_ingest_time").show(5)

---
## 6. Summary & Next Steps

| Component | Destination / Result |
|-----------|----------------------|
| Source | Kafka Topic `trips-raw` |
| Processing | Distributed Spark Streaming (Workers) |
| Target Storage | HDFS Bronze Layer (`/hdfs/bronze/yellow_trips`) |
| Format | Parquet (Columnar Storage) |

### Next Notebook:
`03_spark_batch_silver.ipynb` — Clean, deduplicate and store in **HDFS Silver Layer**.